## Training Block(sample):

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import load_iris

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score

In [2]:
device="cpu"

In [3]:
iris=load_iris()
x=iris.data
y=iris.target

x_train, x_test, y_train, y_test = train_test_split(x, y)

sc=StandardScaler()
x_train=sc.fit_transform(x_train)
x_test=sc.transform(x_test)

x_train=torch.tensor(x_train, dtype=torch.float)
y_train=torch.tensor(y_train, dtype=torch.long)
x_test=torch.tensor(x_test, dtype=torch.float)
y_test=torch.tensor(y_test, dtype=torch.long)

In [4]:
train_dataset=TensorDataset(x_train, y_train)

In [5]:
train_loader=DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

In [6]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.network=nn.Sequential(
            nn.Linear(4, 60),
            nn.ReLU(),
            
            nn.Linear(60, 80),
            nn.ReLU(),
            
            nn.Linear(80, 3)
        )

    def forward(self, x):
        return self.network(x)

In [7]:
model=Classifier().to(device)

In [8]:
criterian=nn.CrossEntropyLoss()

optimizer=optim.Adam(
    model.parameters(),
    lr=0.001
)

In [9]:
epochs=40
for epoch in range(epochs):
    model.train()
    total_loss=0
    
    for x_batch, y_batch in train_loader:
        x_batch=x_batch.to(device)
        y_batch=y_batch.to(device)

        output=model(x_batch)

        loss=criterian(output, y_batch)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss+=loss.item()

        print(
            "epoch:  ", epoch, "/", epochs,"\n",
            "loss:   ", total_loss, "\n\n"
        )

/home/rahaan/Desktop/projects/GESH-ECG-anomaly-deltector/detector_model/VE/lib/python3.10/site-packages/torch/cuda/__init__.py:422: UserWarning: Found GPU0 NVIDIA GeForce MX330 which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Your installed torch==2.13.0+cu130 does not include kernels for this GPU. Reinstall the same version against a CUDA build that does, e.g.:
  For CUDA 12.6 use pip install torch==2.13.0 --index-url https://download.pytorch.org/whl/cu126
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/rahaan/Desktop/projects/GESH-ECG-anomaly-deltector/detector_model/VE/lib/py

epoch:   0 / 40 
 loss:    1.0606971979141235 


epoch:   0 / 40 
 loss:    2.111962914466858 


epoch:   0 / 40 
 loss:    3.126050591468811 


epoch:   0 / 40 
 loss:    4.136083126068115 


epoch:   0 / 40 
 loss:    5.138305187225342 


epoch:   0 / 40 
 loss:    6.093963921070099 


epoch:   0 / 40 
 loss:    7.0241371393203735 


epoch:   1 / 40 
 loss:    0.9087696075439453 


epoch:   1 / 40 
 loss:    1.8451417088508606 


epoch:   1 / 40 
 loss:    2.718452274799347 


epoch:   1 / 40 
 loss:    3.549173414707184 


epoch:   1 / 40 
 loss:    4.424775004386902 


epoch:   1 / 40 
 loss:    5.237573325634003 


epoch:   1 / 40 
 loss:    6.030832350254059 


epoch:   2 / 40 
 loss:    0.8419807553291321 


epoch:   2 / 40 
 loss:    1.6626222729682922 


epoch:   2 / 40 
 loss:    2.341071307659149 


epoch:   2 / 40 
 loss:    3.084500014781952 


epoch:   2 / 40 
 loss:    3.780974566936493 


epoch:   2 / 40 
 loss:    4.44303160905838 


epoch:   2 / 40 
 loss:    5.194225

In [10]:
x_test=x_test.to(device)
model.eval()
with torch.no_grad():
    outputs=model(x_test)

y_pred=torch.argmax(outputs, dim=1)
print(y_pred)

tensor([1, 0, 1, 1, 0, 0, 1, 2, 2, 0, 0, 1, 2, 2, 0, 2, 1, 2, 0, 2, 2, 1, 1, 0,
        2, 0, 0, 0, 2, 2, 1, 2, 2, 0, 0, 2, 0, 0])


In [11]:
print(y_test)

tensor([1, 0, 1, 1, 0, 0, 1, 2, 1, 0, 0, 1, 2, 1, 0, 2, 1, 2, 0, 2, 1, 1, 1, 0,
        2, 0, 0, 0, 2, 2, 1, 2, 2, 0, 0, 2, 0, 0])


In [12]:
print(accuracy_score(y_test, y_pred))

0.9210526315789473


In [13]:
for i in range(len(y_test)):
    if y_test[i]!=y_pred[i]:
        print("i: ", i, "     ", y_test[i], "  ", y_pred[i])

i:  8       tensor(1)    tensor(2)
i:  13       tensor(1)    tensor(2)
i:  20       tensor(1)    tensor(2)
